In [ ]:
# Combined Example: All 5 Techniques in One Prompt

PROMT = """
# Technique 4: Role Assignment
You are a senior security engineer specializing in
web application security (OWASP Top 10).

# Technique 1: Explicit Criteria
Analyze this HTTP endpoint for security vulnerabilities.

Evaluate ONLY these categories:
1. Injection (SQL, XSS, command injection)
2. Broken Authentication
3. Sensitive Data Exposure
4. Missing Rate Limiting

For each vulnerability found, provide:
- Severity: CRITICAL / HIGH / MEDIUM / LOW
- CWE number (if applicable)
- Specific line reference
- Exploitation scenario (one sentence)
- Remediation code

If a category has no issues, state "No issues found."
Do not invent vulnerabilities.

# Technique 5: Structured Output
Return your analysis as JSON matching this schema:
{
  "vulnerabilities": [{
    "category": "string",
    "severity": "CRITICAL|HIGH|MEDIUM|LOW",
    "cwe": "CWE-xxx or null",
    "line": "number",
    "description": "string",
    "exploitation": "string",
    "remediation": "string (code)"
  }],
  "overall_risk": "CRITICAL|HIGH|MEDIUM|LOW",
  "summary": "string (max 50 words)"
}

# Technique 2: Few-Shot Example
Example output for a SQL injection vulnerability:
{
  "category": "Injection",
  "severity": "CRITICAL",
  "cwe": "CWE-89",
  "line": 15,
  "description": "User input concatenated into SQL query",
  "exploitation": "Attacker can extract entire database",
  "remediation": "Use parameterized queries"
}

# Technique 3: Chain-of-Thought
Before producing the JSON, think through each
endpoint parameter and trace how user input flows
through the code to identify taint paths.

Code to analyze:
[paste code here]
"""

In [ ]:
# Complete Example: Criteria-Driven Code Review

PROMT = """
Review this pull request diff for a Python FastAPI backend.

Evaluate across these 4 categories (in order):

## 1. Correctness
- Logic errors that would cause incorrect behavior
- Unhandled exceptions that would crash the process
- Race conditions in async code
- Missing await keywords on async functions

## 2. Security
- SQL injection (raw string interpolation in queries)
- Path traversal (user input in file paths)
- Missing authentication/authorization checks
- Sensitive data exposure in logs or error messages
- Hardcoded secrets or credentials

## 3. Performance
- N+1 query patterns (queries inside loops)
- Missing database indexes for frequent query patterns
- Unbounded result sets (no LIMIT clause)
- Blocking I/O in async handlers

## 4. Maintainability
- Functions longer than 50 lines
- Missing type annotations on public functions
- Unclear variable names (single letters, abbreviations)
- Duplicated logic that should be extracted

For each issue:
1. Quote the exact line(s) from the diff
2. Explain the risk in one sentence
3. Rate severity: CRITICAL / HIGH / MEDIUM / LOW
4. Provide a corrected code snippet

If no issues are found in a category, state:
"No issues found in [category]."

Do NOT invent issues. Only report problems you can
demonstrate with specific code references.

At the end, provide:
- Overall assessment: APPROVE / REQUEST_CHANGES / BLOCK
- Summary: 2-3 sentences explaining your recommendation
"""

In [ ]:
# Example: Full Classification Prompt 
PROMT = """
Classify this customer feedback.

Return JSON with these fields:
- category: exactly one of BUG | FEATURE_REQUEST | PRAISE
           | COMPLAINT | QUESTION
- confidence: high | medium | low
- sentiment: 1 (very negative) to 5 (very positive)
- summary: max 15 words

Example:
Input: "Your app crashes every time I try to upload a PDF.
This is so frustrating!"
Output: {"category": "BUG", "confidence": "high",
"sentiment": 1, "summary": "App crashes on PDF upload,
user frustrated"}

Example:
Input: "Just wanted to say the new search feature is
incredible. Saves me hours every week!"
Output: {"category": "PRAISE", "confidence": "high",
"sentiment": 5, "summary": "User loves new search feature,
significant time savings"}

Example:
Input: "Can you add Slack integration? Our team lives in
Slack and switching tools is painful."
Output: {"category": "FEATURE_REQUEST", "confidence": "high",
"sentiment": 2, "summary": "Requests Slack integration due
to workflow friction"}

Now classify this feedback:
[paste feedback here]
"""

In [ ]:
# Chain-of-Thought & Extended Thinking
PROMT = """
A store has 3 shelves. Each shelf holds 4 boxes.
Each box contains 6 red balls and 2 blue balls.
If I remove all blue balls from the top shelf,
how many balls remain in the entire store?

Think step by step before giving your final answer.
"""

# Structured CoT: Beyond "Think Step by Step"
PROMT = """
Should we use a message queue or direct API calls for
our microservice communication?

Analyze this decision using this framework:

## 1. Requirements Analysis
- What are the throughput requirements?
- What are the latency requirements?
- Is ordering important?
- Is exactly-once delivery required?

## 2. Evaluate Option A: Message Queue
- Pros (with specific technical reasoning)
- Cons (with specific technical reasoning)
- Best suited when: [conditions]

## 3. Evaluate Option B: Direct API Calls
- Pros (with specific technical reasoning)
- Cons (with specific technical reasoning)
- Best suited when: [conditions]

## 4. Recommendation
- State your recommendation
- Justify it against the requirements from step 1
- Note what would change your recommendation
"""


#### Structured Output Patterns: using a tool to structure the output format
```
// Define a tool with your desired output schema
tools: [{
  name: "extract_invoice_data",
  description: "Extract structured data from an invoice",
  input_schema: {
    type: "object",
    properties: {
      invoice_number: { type: "string" },
      date: { type: "string", format: "date" },
      vendor_name: { type: "string" },
      line_items: {
        type: "array",
        items: {
          type: "object",
          properties: {
            description: { type: "string" },
            quantity: { type: "number" },
            unit_price: { type: "number" },
            total: { type: "number" }
          },
          required: ["description", "quantity",
                     "unit_price", "total"]
        }
      },
      total: { type: "number" }
    },
    required: ["invoice_number", "vendor_name",
               "total"]
  }
}]

// Force Claude to call the tool
tool_choice: { type: "any" }
```

##### The 5 Rules of Effective Tool Descriptions
** Explain WHAT the Tool Does**
The tool name alone is not enough. "search_database" could mean full-text search, SQL query execution, or vector similarity search. The description should make the tool's purpose unambiguous: "Executes a read-only SQL query against the PostgreSQL analytics database. Returns results as an array of row objects."

** Explain WHEN to Use It **
Differentiate this tool from similar tools. "Use this tool when the user asks about historical data or trends. For real-time metrics, use the 'get_live_dashboard' tool instead." Without this guidance, Claude will guess when tools overlap.

** Define Input Formats with Examples **
Do not just say "query: string." Say "query: A SQL SELECT statement. Example: 'SELECT name, revenue FROM customers WHERE region = 'EMEA' ORDER BY revenue DESC LIMIT 10'." Examples in descriptions are just as powerful as examples in prompts.


** Describe Output Format **
"Returns a JSON object with 'results' (array of rows), 'row_count' (integer), and 'execution_time_ms' (integer). If no results match, returns an empty array -- not null." This prevents Claude from making incorrect assumptions about the return format.

** Note Edge Cases and Limitations **
"Maximum 1000 rows per query. Queries that take longer than 30 seconds will timeout. This tool only has read access -- INSERT, UPDATE, and DELETE statements will return an error." These boundaries prevent Claude from attempting impossible operations.



In [ ]:
# Logging Tool Selection for A/B Testing
# Log every tool selection for analysis
function logToolSelection(query, selectedTool, expectedTool) {
  var entry = {
    timestamp: new Date().toISOString(),
    query: query,
    selected: selectedTool,
    expected: expectedTool,
    correct: selectedTool === expectedTool,
    description_version: toolDescriptionVersion
  };
  analytics.log("tool_selection", entry);
}

# After collecting 100+ entries, analyze accuracy
# If accuracy < 95% for a tool, revise its description
# Focus on the WHEN clause -- that is where most
#misrouting originates

In [ ]:
# Review Pattern: Independent Reviewer Pattern

# Step 1: Generate code in Session A
def generatedCode = await claude.generate({
  "prompt": "Write a function that validates email addresses..."
});

#Step 2: Review in Session B (fresh context)
review = await claude.generate({
"prompt": """Review this code. You did NOT write this code.
Evaluate it objectively for:
1. Correctness (does it handle all valid RFC 5322 emails?)
2. Security (ReDoS vulnerability in regex?)
3. Edge cases (empty string, null, unicode, very long input)
4. Performance (is the regex efficient?)

Code to review:
\`\`\`
${generatedCode}
\`\`\`

For each issue, show the problematic line and a fix.
If the code is correct for a category, say "No issues found."
"""
});

#### Key Takeaways
**Explicit criteria is the highest-leverage technique**
Every underperforming prompt can be improved by adding more specific criteria. Define what to include, how to format it, quality standards, and edge case handling. This single technique accounts for more improvement than all others combined.

**Few-shot examples: 2-3 is optimal**
Show, do not describe. Choose examples that cover the common case, an edge case, and optionally a negative case. Quality matters more than quantity. Pull from real data when possible.

**Chain-of-Thought is not always better**
Use CoT for multi-step reasoning, math, debugging, and analysis. Skip it for simple factual recall and classification. Extended Thinking is the more powerful alternative for genuinely complex problems.    

**Use tool_use for guaranteed structured output**
JSON Schema + tool_choice: "any" guarantees valid JSON matching your schema. Combine with a validation-retry loop for business logic that JSON Schema cannot express.

**Tool descriptions determine tool selection accuracy**
Explain what the tool does, when to use it (vs alternatives), input formats with examples, output format, and edge cases. The #1 cause of misrouting is overlapping descriptions.

**Multi-pass review catches what self-review misses**
An independent Claude instance is less biased than self-review. For large codebases, use specialized passes (local + integration) to avoid attention dilution.